# verify 명령 테스트 (작업지시서 §8.1)

`convert`가 패키지를 **만드는** 쪽이라면, `verify`는 만들어진 패키지가 **계약을
지키는지 검사**하는 쪽입니다. M1 범위는 두 가지:

- **V1 스키마** — `schemas/{meta,references,diagnostics}.schema.json`으로 세 결정론
  산출물을 검증(엄격: 선언 안 된 키가 있으면 실패). `cells.jsonl`은 스키마 대상이
  아니라 각 줄 JSON 파싱만 봅니다. `semantics`는 M3라 있으면 생략.
- **V3 재현성** — `--source` 원본을 주면 임시 폴더로 **다시 변환해** 결정론 계층이
  같은지 비교(meta는 `generated_at` 제외). 원본이 없으면 **실패가 아니라 생략**,
  V1만으로 통과. 원본을 줬는데 다르면 verify 실패.

확인하는 것:
1. **스키마 3종** 존재·로드
2. **32종 전수 V1** — 모든 실산출이 스키마를 통과(xlsx/xls 차이 포함)
3. **xls 변형** — count=null·unavailable_xls·format_limitations 문자열·hidden.sheets가
   실제로 통과하는지
4. **V3 재현성** — 원본 주면 통과 / 다른 원본이면 실패
5. **부정 케이스** — 선언 안 된 키(엄격)·필수 파일 누락이면 실패
6. **exit code 계약** — 통과 0 / 실패 1 / 미구현 스텁 2

> 실행 전 커널이 이 프로젝트의 `.venv`인지 확인하세요.

In [1]:
# 0. 준비
import json, os, shutil, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from excel_to_skill.cli import _convert_one
from excel_to_skill.meta import _converter_version
from excel_to_skill.verify import verify_package, Check

RAW = ROOT / "workingpaper_raw"
SANDBOX = ROOT / "tests" / "_output" / "verify_nb"
if SANDBOX.exists():
    shutil.rmtree(SANDBOX)
CONV = SANDBOX / "converted"
CV = _converter_version()
ENV = {**os.environ, "PYTHONPATH": str(ROOT / "src")}

def run(*a):
    p = subprocess.run([sys.executable, "-m", "excel_to_skill.cli", *a],
                       cwd=str(ROOT), env=ENV, capture_output=True, text=True)
    return p.returncode, p.stdout, p.stderr

print("루트:", ROOT, "| converter:", CV)

루트: /home/shin/Project/Excel_to_Skill | converter: 0.1.0


## 1. 스키마 3종 존재·로드

`schemas/`에 meta·references·diagnostics 스키마가 있고 JSON으로 읽히는지. semantics는
M3라 아직 없어도 정상입니다.

In [2]:
SCHEMAS = ROOT / "schemas"
for n in ["meta.schema.json", "references.schema.json", "diagnostics.schema.json"]:
    d = json.loads((SCHEMAS / n).read_text(encoding="utf-8"))
    strict = d.get("additionalProperties") is False
    print(f"  O {n:32} 로드 OK · 엄격(additionalProperties=false)={strict}")
    assert strict, "엄격 스키마여야"
print("semantics.schema.json 존재:", (SCHEMAS / "semantics.schema.json").exists(), "(M3 예정)")
print("\n결과: 통과")

  O meta.schema.json                 로드 OK · 엄격(additionalProperties=false)=True
  O references.schema.json           로드 OK · 엄격(additionalProperties=false)=True
  O diagnostics.schema.json          로드 OK · 엄격(additionalProperties=false)=True
semantics.schema.json 존재: False (M3 예정)

결과: 통과


## 2. 32종 전수 V1 — 모든 실산출이 스키마를 통과하나

코퍼스 32종을 전부 변환하고 각 패키지에 V1(+필수파일·cells 파싱)을 돌립니다.
스키마가 실제 방출과 한 군데라도 어긋나면 여기서 실패로 잡힙니다.

In [3]:
CONV.mkdir(parents=True)
rows, fails = [], []
for f in sorted(RAW.glob("*.xls*")):
    pkg = _convert_one(f, CONV, force=True, cv=CV)
    res = verify_package(pkg)                # 원본 미제공 → V3 생략
    bad = [c for c in res.checks if not c.ok and not c.skipped]
    fmt = json.loads((pkg / "meta.json").read_text())["source"]["format"]
    rows.append((f.name[:36], fmt, "OK" if res.ok else "FAIL"))
    if bad:
        fails.append((f.name, [(c.name, c.detail) for c in bad]))

print(f"{'파일':<38} {'fmt':<5} V1")
for name, fmt, v in rows:
    print(f"{name:<38} {fmt:<5} {v}")
print(f"\n전수 {len(rows)}종 · V1 실패 {len(fails)}")
for name, bad in fails:
    print("  X", name, bad)
assert not fails, "스키마 불일치 존재"
print("결과: 통과 (32종 전수 V1)")

[cache miss:force] 감사조서서식_01 조서파일 KIFRS.xlsx → 생성
[cache miss:force] 감사조서서식_02 영구조서목록.xlsx → 생성
[cache miss:force] 감사조서서식_03 일반조서목록 KIFRS.xlsx → 생성
[cache miss:force] 감사조서서식_04 감사조서철 작성 및 보존.xlsx → 생성
[cache miss:force] 감사조서서식_1100A_계약전 위험평가표 수임 2025.xlsx → 생성


[cache miss:force] 감사조서서식_1100~1300 감사계약.xlsx → 생성


[cache miss:force] 감사조서서식_1101A_계약전 위험평가표 계속 2025.xlsx → 생성


[cache miss:force] 감사조서서식_1300A_독립성준수검토조서 2025.xlsx → 생성


[cache miss:force] 감사조서서식_1300B_개별 감사 참여자의 독립성 준수 확인서 2025.xlsx → 생성


...

[cache miss:force] 감사조서서식_7540 연결공시사항점검표 (KIFRS용)_2025.xls → 생성


[cache miss:force] 감사조서서식_7555 연결심리사항점검표.xlsx → 생성
[cache miss:force] 감사조서서식_7555A_그룹감사 업무수행이사와 논의한 사항 및 자문한 사항 2025 신규.xlsx → 생성
[cache miss:force] 감사조서서식_8100~8700 감사완결 2025.xlsx → 생성


[cache miss:force] 감사조서서식_8400 공시사항점검표 (KIFRS용)_2025.xls → 생성


[cache miss:force] 감사조서서식_8550 심리사항점검표.xlsx → 생성
[cache miss:force] 감사조서서식_8550A_업무수행이사와 논의한 사항 및 자문한 사항 2025 신규.xlsx → 생성
[cache miss:force] 감사조서서식_9100~9800 내부회계관리제도감사 조서서식(예시).xlsx → 생성


파일                                     fmt   V1
감사조서서식_01 조서파일 KIFRS.xlsx              xlsx  OK
감사조서서식_02 영구조서목록.xlsx                  xlsx  OK
감사조서서식_03 일반조서목록 KIFRS.xlsx            xlsx  OK
감사조서서식_04 감사조서철 작성 및 보존.xlsx           xlsx  OK
감사조서서식_1100A_계약전 위험평가표 수임 2025.xlsx    xlsx  OK
감사조서서식_1100~1300 감사계약.xlsx             xlsx  OK
감사조서서식_1101A_계약전 위험평가표 계속 2025.xlsx    xlsx  OK
감사조서서식_1300A_독립성준수검토조서 2025.xlsx       xlsx  OK
감사조서서식_1300B_개별 감사 참여자의 독립성 준수 확인서 2   xlsx  OK
감사조서서식_1300C_산업전문가검토조서.xlsx            xlsx  OK
감사조서서식_2100-2700 위험평가 2025.xlsx        xlsx  OK
감사조서서식_2110-2 계약후 감사위험 재평가 2025.xlsx   xlsx  OK
감사조서서식_2700A 중요성요약표 및 산정 template 20   xlsx  OK
감사조서서식_3100-3800 위험에 대한 대응 2025.xlsx   xlsx  OK
감사조서서식_3650 감사 전 재무제표 확인.xlsx          xlsx  OK
감사조서서식_3900_3900A 핵심감사사항.xlsx          xlsx  OK
감사조서서식_3910 수주산업 핵심감사사항.xlsx           xlsx  OK
감사조서서식_4000 계정별 실증절차 (KIFRS용) 2025.x   xlsx  OK
감사조서서식_4000P-1 K-IFRS 제1115호 고객과의 계약   xlsx  OK
감사조서서식_7000~7570 그룹감사 2025.xlsx        x

## 3. xls 변형 — 다른 모양이 실제로 통과하나

`.xls`는 xlsx와 산출 모양이 다릅니다: 외부링크 `count=null`(관찰 불가≠0),
`observability.workbook="unavailable_xls"` + `note` 문자열, `format_limitations`
문자열, 숨김 시트가 `hidden.sheets`에. 이 케이스들이 스키마를 통과하는지 콕 집어 봅니다.

In [4]:
xls_pkgs = [p for p in CONV.iterdir() if p.is_dir()
            and json.loads((p / "meta.json").read_text())["source"]["format"] == "xls"]
print("xls 패키지 수:", len(xls_pkgs))
for pkg in sorted(xls_pkgs):
    diag = json.loads((pkg / "data/diagnostics.json").read_text())
    refs = json.loads((pkg / "data/references.json").read_text())
    print(f"\n{pkg.name[:40]}")
    print("  external_links.count :", diag["external_links"]["count"])
    print("  observability.workbook:", refs["observability"]["workbook"])
    print("  observability.note 타입:", type(refs["observability"]["note"]).__name__)
    print("  format_limitations 타입:", type(diag["format_limitations"]).__name__)
    print("  hidden.sheets 개수    :", len(diag["hidden"]["sheets"]))
    assert diag["external_links"]["count"] is None
    assert refs["observability"]["workbook"] == "unavailable_xls"
    assert isinstance(diag["format_limitations"], str)
    assert verify_package(pkg).ok
print("\n결과: 통과 (xls 변형도 V1 통과)")

xls 패키지 수: 2

감사조서서식_7540_연결공시사항점검표_(KIFRS용)_2025_483a
  external_links.count : None
  observability.workbook: unavailable_xls
  observability.note 타입: str
  format_limitations 타입: str
  hidden.sheets 개수    : 2



감사조서서식_8400_공시사항점검표_(KIFRS용)_2025_dda5f4
  external_links.count : None
  observability.workbook: unavailable_xls
  observability.note 타입: str
  format_limitations 타입: str
  hidden.sheets 개수    : 2



결과: 통과 (xls 변형도 V1 통과)


## 4. V3 재현성 — 원본 주면 통과 / 다른 원본이면 실패

`--source`로 원본을 주면 다시 변환해 결정론 계층을 비교합니다. 같은 원본이면 통과,
엉뚱한 원본을 주면 sha가 달라 **실패**여야 합니다.

In [5]:
SRC = next(RAW.glob("*1100~1300*"))
pkg = next(p for p in CONV.iterdir() if p.name.startswith("감사조서서식_1100~1300"))

# 올바른 원본 → V3 통과
res = verify_package(pkg, source=SRC)
v3 = next(c for c in res.checks if c.name == "V3")
print("올바른 원본 → V3:", "PASS" if v3.ok and not v3.skipped else "??", "|", v3.detail)
assert res.ok and v3.ok and not v3.skipped

# 다른 원본 → V3 실패
OTHER = next(RAW.glob("*7540*"))
res2 = verify_package(pkg, source=OTHER)
v3b = next(c for c in res2.checks if c.name == "V3")
print("다른 원본   → V3:", "FAIL" if not v3b.ok else "??", "|", v3b.detail)
assert not res2.ok and not v3b.ok
print("\n결과: 통과 (원본 일치=재현 확인 / 불일치=거부)")

[cache miss:force] 감사조서서식_1100~1300 감사계약.xlsx → 생성


올바른 원본 → V3: PASS | 재변환 결과 동일(결정론)
다른 원본   → V3: FAIL | --source가 이 패키지의 원본과 다름(sha256 불일치)

결과: 통과 (원본 일치=재현 확인 / 불일치=거부)


## 5. 부정 케이스 — 엄격 스키마·필수 파일 누락

패키지를 복제해 일부러 망가뜨립니다: (a) diagnostics에 선언 안 된 키를 넣으면 엄격
스키마가 잡아야 하고, (b) 필수 파일을 지우면 파일 검사가 실패해야 합니다.

In [6]:
base = next(p for p in CONV.iterdir() if p.name.startswith("감사조서서식_1100~1300"))

# (a) 선언 안 된 키 주입
broke = SANDBOX / "broke_extra"
shutil.copytree(base, broke)
dp = broke / "data/diagnostics.json"
d = json.loads(dp.read_text()); d["unexpected_key"] = 1
dp.write_text(json.dumps(d, ensure_ascii=False, indent=2), encoding="utf-8")
res = verify_package(broke)
c = next(c for c in res.checks if c.name == "V1:data/diagnostics.json")
print("(a) 여분 키 → V1:diagnostics:", "FAIL" if not c.ok else "??")
print("     사유:", c.detail)
assert not res.ok and not c.ok

# (b) 필수 파일 삭제
broke2 = SANDBOX / "broke_missing"
shutil.copytree(base, broke2)
(broke2 / "data/cells.jsonl").unlink()
res2 = verify_package(broke2)
cf = next(c for c in res2.checks if c.name == "files")
print("(b) cells.jsonl 삭제 → files:", "FAIL" if not cf.ok else "??", "|", cf.detail)
assert not res2.ok and not cf.ok
print("\n결과: 통과 (엄격 스키마·누락 모두 잡음)")

(a) 여분 키 → V1:diagnostics: FAIL
     사유: (root): Additional properties are not allowed ('unexpected_key' was unexpected)
(b) cells.jsonl 삭제 → files: FAIL | 누락: ['data/cells.jsonl']

결과: 통과 (엄격 스키마·누락 모두 잡음)


## 6. exit code 계약 (실제 CLI)

verify 명령이 **exit code로** 판정을 알리는지 subprocess로 확인: 통과 0, 실패 1,
그리고 아직 스텁인 `annotate`는 2.

In [7]:
# 통과(원본 미제공 V1만) → 0
rc, out, _ = run("verify", str(base))
print("verify(정상)        exit:", rc)
assert rc == 0 and "통과" in out

# 실패(여분 키 패키지) → 1
rc, out, err = run("verify", str(broke))
print("verify(스키마 위반) exit:", rc)
assert rc == 1 and "실패" in out

# 원본 불일치 → 1
rc, _, _ = run("verify", str(base), "--source", str(next(RAW.glob("*7540*"))))
print("verify(원본 불일치) exit:", rc)
assert rc == 1

# 스텁 → 2
rc, _, err = run("annotate", "x")
print("annotate(스텁)      exit:", rc, "|", err.strip())
assert rc == 2
print("\n결과: 통과 (exit code 권위)")

verify(정상)        exit: 0


verify(스키마 위반) exit: 1


verify(원본 불일치) exit: 1
annotate(스텁)      exit: 2 | 'annotate' 은(는) 아직 구현되지 않았습니다 (M3 단계).

결과: 통과 (exit code 권위)
